# NAICEWS Cascade Model Training - Google Colab Version

This notebook trains the cascade prediction model that evaluates whether upstream anomalies trigger downstream cascade events.

## Instructions:
1. Upload your data files to Colab:
   - `data/processed/master_multi_city_4h.parquet`
   - `data/models/causal_graph.json`
   - `data/raw/cities_metadata.json`
2. Run all cells sequentially
3. Download the trained model: `cascade_predictor.pkl`
4. Place it in your project: `data/models/cascade_predictor.pkl`

In [ ]:
# Install required packages
!pip install pandas numpy scikit-learn xgboost joblib pyarrow -q

In [ ]:
# Import libraries
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import classification_report, mean_squared_error
import joblib
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

In [ ]:
# Upload files instruction
print("Please upload the following files to Colab:")
print("1. master_multi_city_4h.parquet")
print("2. causal_graph.json")
print("3. cities_metadata.json")
print("\nUse the file upload button on the left sidebar.")

In [ ]:
# Load data
print("Loading data...")
df = pd.read_parquet("/content/master_multi_city_4h.parquet")
with open("/content/causal_graph.json", 'r') as f:
    causal_graph = json.load(f)
with open("/content/cities_metadata.json", 'r') as f:
    cities_metadata = json.load(f)

print(f"Loaded {len(df)} rows")
print(f"Loaded {len(causal_graph['edges'])} causal edges")
print(f"Loaded {len(cities_metadata)} cities")

In [ ]:
# Prepare training data for each causal edge
print("Preparing training data...")

edges = causal_graph['edges']
city_lookup = {city['id']: city for city in cities_metadata}

training_samples = []

for edge in edges:
    source = edge['source_node']
    target = edge['target_node']
    delay_hours = edge['delay_hours']
    
    # Get data for source and target cities
    source_data = df[df['city_id'] == source].sort_values('time_4h').reset_index(drop=True)
    target_data = df[df['city_id'] == target].sort_values('time_4h').reset_index(drop=True)
    
    if source_data.empty or target_data.empty:
        continue
    
    # Calculate PM2.5 Z-score for source
    source_mean = source_data['pm2_5'].mean()
    source_std = source_data['pm2_5'].std()
    source_data['pm2_5_zscore'] = (source_data['pm2_5'] - source_mean) / (source_std + 1e-8)
    
    # Create training samples
    delay_steps = delay_hours // 4  # Convert hours to 4-hour steps
    
    for i in range(len(source_data) - delay_steps):
        # Source features at time t
        source_row = source_data.iloc[i]
        
        # Target features at time t + delay
        target_row = target_data.iloc[i + delay_steps] if i + delay_steps < len(target_data) else None
        
        if target_row is None:
            continue
        
        # Input features
        features = {
            'source_pm25': source_row['pm2_5'],
            'source_pm25_zscore': source_row['pm2_5_zscore'],
            'source_wind_speed': source_row['wind_speed'],
            'source_wind_u': source_row['wind_u'],
            'source_wind_v': source_row['wind_v'],
            'source_inversion_proxy': source_row['inversion_proxy'],
            'target_pm25_baseline': target_row['pm2_5'],
            'target_humidity': target_row['humidity'],
            'pressure_gradient': source_row['surface_pressure'] - target_row['surface_pressure'],
            'correlation_strength': edge['correlation_coefficient'],
            'delay_hours': delay_hours
        }
        
        # Target: will target have severe spike at t + delay?
        target_severe = 1 if target_row['pm2_5'] > 120 else 0
        target_delta = target_row['pm2_5'] - source_row['pm2_5']
        
        features['target_severe'] = target_severe
        features['target_delta'] = target_delta
        
        training_samples.append(features)

train_df = pd.DataFrame(training_samples)
print(f"Created {len(train_df)} training samples")

In [ ]:
# Split features and targets
feature_cols = ['source_pm25', 'source_pm25_zscore', 'source_wind_speed', 
                'source_wind_u', 'source_wind_v', 'source_inversion_proxy',
                'target_pm25_baseline', 'target_humidity', 'pressure_gradient',
                'correlation_strength', 'delay_hours']

X = train_df[feature_cols].values
y_class = train_df['target_severe'].values
y_reg = train_df['target_delta'].values

# Split into train/test
X_train, X_test, y_class_train, y_class_test, y_reg_train, y_reg_test = train_test_split(
    X, y_class, y_reg, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

In [ ]:
# Train classification model (predict severe spike)
print("Training classification model...")
classifier = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

classifier.fit(X_train, y_class_train)

# Evaluate
y_class_pred = classifier.predict(X_test)
print("\nClassification Report:")
print(classification_report(y_class_test, y_class_pred))

In [ ]:
# Train regression model (predict PM2.5 delta)
print("Training regression model...")
regressor = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

regressor.fit(X_train, y_reg_train)

# Evaluate
y_reg_pred = regressor.predict(X_test)
mse = mean_squared_error(y_reg_test, y_reg_pred)
print(f"\nRegression MSE: {mse:.2f}")
print(f"Regression RMSE: {np.sqrt(mse):.2f}")

In [ ]:
# Save models
print("\nSaving models...")

cascade_model = {
    'classifier': classifier,
    'regressor': regressor,
    'feature_cols': feature_cols
}

joblib.dump(cascade_model, '/content/cascade_predictor.pkl')
print("Saved cascade_predictor.pkl")

In [ ]:
# Download the model
from google.colab import files

print("\nDownloading cascade_predictor.pkl...")
files.download('/content/cascade_predictor.pkl')

print("\n✅ Model training completed!")
print("\nNext steps:")
print("1. Download cascade_predictor.pkl")
print("2. Place it in your project: data/models/cascade_predictor.pkl")
print("3. Run: python scripts/airshed_pipeline.py")